# 08 — Interpretive Synthesis and Cases for Close Reading

Notebook 05 measures the degree, geometry, shape, participants, and prompt trajectory of translation disagreement. Notebooks 06 and 07 then examine the rhetoric and definitional content of model rationales. This notebook brings those layers back together for interpretation.

The goal is not to assign every language a single causal category. Instead, I use overlapping evidence to identify cases where close reading is most valuable: fragmented translations, service outliers, prompt sensitivity, script disagreement, source retention, machine-translation/LLM splits, Wikipedia evidence, and explicit model claims about borrowing, coinage, established equivalents, or uncertainty.

Rationales remain generated claims rather than linguistic validation. The final case-selection score is only a sorting device for scholarly review.


## 8.1 Load the Synthesis Evidence

I load the reusable `disagreement_patterns.csv` produced by `build_disagreement_patterns.py`. Notebook 05 is responsible for rebuilding that table after translation or embedding inputs change; this notebook reads the resulting evidence layer without rerunning the quantitative analysis.


In [1]:
import os
import sys
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd

alt.data_transformers.enable('vegafusion')
PROJECT_ROOT = Path('..')
if not (PROJECT_ROOT / 'scripts' / 'utils.py').exists():
    PROJECT_ROOT = Path('.')
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.utils import read_csv_file, save_chart

DATA_DIR = PROJECT_ROOT / "datasets"
TERM_SLUG = 'digital_humanities'
EVAL_DIR = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')
PATTERN_PATH = os.path.join(EVAL_DIR, 'disagreement_patterns.csv')
FIGURE_DIR = PROJECT_ROOT / 'figures' / 'dhq_article'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

patterns = read_csv_file(PATTERN_PATH)
assert len(patterns) == 881

evidence = patterns.copy()
evidence['has_source_retention'] = evidence['source_retention_service_count'].gt(0)
evidence['has_singleton_outlier'] = evidence['singleton_outlier_service_count'].gt(0)
evidence['has_prompt_instability'] = evidence['prompt_sensitive_service_count'].gt(0)
signal_map = {
    'has_wikipedia': 'Wikipedia term',
    'has_source_retention': 'Source retention',
    'has_script_disagreement': 'Script disagreement',
    'has_mt_llm_split': 'MT/LLM split',
    'has_singleton_outlier': 'Singleton outlier',
    'has_prompt_instability': 'Prompt instability',
}
print(f'{len(patterns):,} language-code rows loaded for interpretive synthesis')


Retrieving translation pipeline data directory path...

881 language-code rows loaded for interpretive synthesis


## 8.2 What the Rationales Claim

Rationale signals are aggregated across the English-language prompt variants. They remain claims made by models, not verification. I distinguish definitional registers from translation stances such as asserting an established equivalent, recommending borrowing, proposing coinage, or admitting uncertainty.

In [2]:
rationale_cols = [
    'claims_established_equivalent', 'claims_borrowing',
    'claims_coinage', 'admits_uncertainty',
    'register_compositional_literal', 'register_scholarly_methodological',
    'register_human_centered', 'register_knowledge_generic',
]
rationale_summary = patterns[rationale_cols].mean().sort_values(ascending=False).rename('share_of_rationales').reset_index(name='share_of_rationales').rename(columns={'index': 'signal'})
rationale_summary['label'] = rationale_summary['signal'].str.replace('_', ' ').str.title()
display(rationale_summary)

rationale_chart = alt.Chart(rationale_summary).mark_bar().encode(
    y=alt.Y('label:N', sort='-x', title=None),
    x=alt.X('share_of_rationales:Q', title='Mean share of English rationales', axis=alt.Axis(format='%')),
    color=alt.Color('signal:N', legend=None, scale=alt.Scale(scheme='tableau10')),
).properties(width=620, height=280, title='Definitional and translation stances in model rationales')
display(rationale_chart)
save_chart(rationale_chart, str(FIGURE_DIR / 'rationale_stances_in_disagreement_analysis.png'))

,signal,share_of_rationales,label
0,register_compositional_literal,0.644271,Register Compositional Literal
1,register_knowledge_generic,0.302861,Register Knowledge Generic
2,register_scholarly_methodological,0.253653,Register Scholarly Methodological
3,claims_borrowing,0.236724,Claims Borrowing
4,register_human_centered,0.156625,Register Human Centered
5,claims_established_equivalent,0.125357,Claims Established Equivalent
6,claims_coinage,0.096165,Claims Coinage
7,admits_uncertainty,0.062970,Admits Uncertainty


alt.Chart(...)

## 8.3 How Rationale Claims Overlap

The prevalence bars do not show whether claims occur in the same rationale. The first heatmap below uses the reviewed rationale-level signal table generated with `disagreement_patterns.csv`. Each cell is the **Jaccard overlap** between two categories: the number of rationales matching both divided by the number matching either. The matrix is symmetric, and the diagonal is 100% by definition. A high value means two regex signals frequently describe the same rationales; it does not mean the claims are equivalent or correct.

The second heatmap connects rationale framing to the external and technical evidence from Notebook 05. Its cells report the percentage-point difference in a claim's mean language-level prevalence when an evidence signal is present versus absent. Positive values mean the claim is more prevalent among languages carrying that signal; negative values mean it is less prevalent. This is descriptive association across language rows, not evidence that the technical signal caused the rationale claim or that either validates the translation. The present-group count appears in the interactive tooltip; especially imbalanced comparisons require caution. For example, 875 languages show prompt instability, leaving only six in the absent group.

In [3]:
RATIONALE_DETAIL_PATH = os.path.join(EVAL_DIR, 'rationale_signal_detail.csv')
rationale_detail = read_csv_file(RATIONALE_DETAIL_PATH)

rationale_labels = {
    'register_compositional_literal': 'Compositional / literal',
    'register_knowledge_generic': 'Generic knowledge',
    'register_scholarly_methodological': 'Scholarly / methodological',
    'claims_borrowing': 'Borrowing',
    'register_human_centered': 'Human-centered',
    'claims_established_equivalent': 'Established equivalent',
    'claims_coinage': 'Coinage',
    'admits_uncertainty': 'Uncertainty',
}
rationale_order = list(rationale_labels)
rationale_label_order = [rationale_labels[column] for column in rationale_order]

for column in rationale_order:
    rationale_detail[column] = (
        rationale_detail[column].astype(str).str.strip().str.lower().eq('true')
    )

overlap_rows = []
for row_signal in rationale_order:
    for column_signal in rationale_order:
        intersection = int((rationale_detail[row_signal] & rationale_detail[column_signal]).sum())
        union = int((rationale_detail[row_signal] | rationale_detail[column_signal]).sum())
        overlap_rows.append({
            'row_claim': rationale_labels[row_signal],
            'column_claim': rationale_labels[column_signal],
            'intersection': intersection,
            'union': union,
            'jaccard': intersection / union if union else np.nan,
        })
overlap_df = pd.DataFrame(overlap_rows)

overlap_base = alt.Chart(overlap_df).encode(
    x=alt.X('column_claim:N', sort=rationale_label_order, title=None, axis=alt.Axis(labelAngle=-42)),
    y=alt.Y('row_claim:N', sort=rationale_label_order, title=None),
    tooltip=['row_claim:N', 'column_claim:N', 'intersection:Q', 'union:Q',
             alt.Tooltip('jaccard:Q', format='.1%', title='Jaccard overlap')],
)
overlap_heat = overlap_base.mark_rect().encode(
    color=alt.Color('jaccard:Q', title='Jaccard overlap',
                    scale=alt.Scale(scheme='blues', domain=[0, 1])),
)
overlap_text = overlap_base.mark_text(fontSize=10).encode(
    text=alt.Text('jaccard:Q', format='.0%'),
    color=alt.condition(alt.datum.jaccard > 0.55, alt.value('white'), alt.value('black')),
)
rationale_overlap_figure = (overlap_heat + overlap_text).properties(
    width=520, height=460,
    title=alt.TitleParams(
        'Rationale-Level Overlap Among Definitional and Translation Claims',
        subtitle='Jaccard overlap: rationales matching both signals / rationales matching either signal.',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)
display(rationale_overlap_figure)
save_chart(rationale_overlap_figure, str(FIGURE_DIR / 'rationale_claim_overlap_heatmap.png'))

def as_bool_signal(series):
    if str(series.dtype) == 'bool':
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({'true', '1', 'yes'})

intersection_rows = []
for evidence_column, evidence_label in signal_map.items():
    present = as_bool_signal(evidence[evidence_column])
    for rationale_column in rationale_order:
        present_mean = patterns.loc[present, rationale_column].mean()
        absent_mean = patterns.loc[~present, rationale_column].mean()
        intersection_rows.append({
            'evidence_signal': evidence_label,
            'rationale_claim': rationale_labels[rationale_column],
            'present_mean': present_mean,
            'absent_mean': absent_mean,
            'difference': present_mean - absent_mean,
            'n_signal_present': int(present.sum()),
        })
claim_evidence_df = pd.DataFrame(intersection_rows)
difference_limit = float(np.ceil(claim_evidence_df['difference'].abs().max() * 20) / 20)

intersection_base = alt.Chart(claim_evidence_df).encode(
    x=alt.X('evidence_signal:N', sort=list(signal_map.values()), title=None,
            axis=alt.Axis(labelAngle=-38)),
    y=alt.Y('rationale_claim:N', sort=rationale_label_order, title=None),
    tooltip=['rationale_claim:N', 'evidence_signal:N', 'n_signal_present:Q',
             alt.Tooltip('present_mean:Q', format='.1%', title='Mean with signal'),
             alt.Tooltip('absent_mean:Q', format='.1%', title='Mean without signal'),
             alt.Tooltip('difference:Q', format='+.1%', title='Difference')],
)
intersection_heat = intersection_base.mark_rect().encode(
    color=alt.Color('difference:Q', title='Difference',
                    scale=alt.Scale(scheme='redblue', reverse=True,
                                    domain=[-difference_limit, difference_limit], domainMid=0)),
)
intersection_text = intersection_base.mark_text(fontSize=10).encode(
    text=alt.Text('difference:Q', format='+.0%'),
    color=alt.condition(f'abs(datum.difference) > {difference_limit * 0.55}',
                        alt.value('white'), alt.value('black')),
)
claim_evidence_figure = (intersection_heat + intersection_text).properties(
    width=620, height=360,
    title=alt.TitleParams(
        'Rationale Claims × External and Technical Evidence',
        subtitle='Percentage-point difference in mean claim prevalence: signal present minus absent.',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)
display(claim_evidence_figure)
save_chart(claim_evidence_figure, str(FIGURE_DIR / 'rationale_claims_by_evidence_signals_heatmap.png'))

alt.LayerChart(...)

alt.LayerChart(...)

## 8.4 Cases for Close Reading

The final score is only a sorting device. It prioritizes rows where several different forms of evidence intersect: fragmentation, service outliers, prompt sensitivity, script disagreement, source retention, MT/LLM division, Wikipedia evidence, and explicit uncertainty or coinage claims. These are strong candidates for qualitative review because no single signal explains them.

In [4]:
review = patterns.copy()
review['review_interest_score'] = (
    review['disagreement_shape'].eq('fragmented_disagreement').astype(int)
    + review['singleton_outlier_service_count'].gt(0).astype(int)
    + review['prompt_sensitive_service_count'].ge(6).astype(int)
    + review['has_script_disagreement'].astype(int)
    + review['source_retention_service_count'].gt(0).astype(int)
    + review['has_mt_llm_split'].astype(int)
    + review['has_wikipedia'].astype(int)
    + review['admits_uncertainty'].gt(0).astype(int)
    + review['claims_coinage'].gt(0).astype(int)
)

review_cols = [
    'language_code', 'language_name', 'language_family', 'comparison_status',
    'disagreement_shape', 'dominant_form_share', 'service_group_pattern',
    'singleton_outlier_services', 'prompt_sensitive_service_count',
    'judge_effect', 'wikipedia_relationship', 'source_retention_service_count',
    'has_script_disagreement', 'has_mt_llm_split', 'claims_borrowing',
    'claims_coinage', 'admits_uncertainty', 'review_interest_score',
]
close_reading_candidates = review.sort_values(
    ['review_interest_score', 'n_unique_raw', 'prompt_sensitive_service_count'],
    ascending=False,
)[review_cols]
display(close_reading_candidates.head(30))

,language_code,language_name,language_family,comparison_status,disagreement_shape,dominant_form_share,service_group_pattern,singleton_outlier_services,prompt_sensitive_service_count,judge_effect,wikipedia_relationship,source_retention_service_count,has_script_disagreement,has_mt_llm_split,claims_borrowing,claims_coinage,admits_uncertainty,review_interest_score
457,lus,Mizo,Sino-Tibetan languages,substantive_variation,fragmented_disagreement,0.400000,api_aligned_local_fragmented,NaN,7,judge_induced_convergence,no_wikipedia_term,6,True,True,0.500000,0.125000,0.083333,7
759,ti,Tigrinya,Afro-Asiatic languages,substantive_variation,fragmented_disagreement,0.100000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,2,True,False,0.166667,0.125000,0.125000,6
664,sat,Santali,Austro-Asiatic languages,substantive_variation,fragmented_disagreement,0.111111,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,2,True,False,0.208333,0.041667,0.041667,6
137,chp,Chipewyan,Athabaskan-Eyak-Tlingit,substantive_variation,fragmented_disagreement,0.125000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,1,True,False,0.083333,0.083333,0.083333,6
184,dtm,Tomo Kan Dogon,Dogon,substantive_variation,fragmented_disagreement,0.125000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,1,True,False,0.291667,0.041667,0.166667,6
216,fan,Fang,Atlantic-Congo,substantive_variation,fragmented_disagreement,0.125000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,1,True,False,0.250000,0.083333,0.083333,6
328,iu,Inuktitut,Eskimo-Aleut languages,substantive_variation,fragmented_disagreement,0.222222,mixed_group_alignment,NaN,8,judge_induced_convergence,no_wikipedia_term,1,True,False,0.125000,0.041667,0.041667,6
351,kcg,Tyap,Atlantic-Congo,substantive_variation,fragmented_disagreement,0.125000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,2,True,False,0.208333,0.041667,0.125000,6
438,lki,Laki,Indo-European languages,substantive_variation,fragmented_disagreement,0.125000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,2,True,False,0.333333,0.125000,0.041667,6
449,lsm,Saamia,Atlantic-Congo,substantive_variation,fragmented_disagreement,0.125000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,1,True,False,0.304348,0.086957,0.043478,6


## 8.5 Interpretation

The central result is not that disagreement belongs to one of several stable types. It is that different dimensions overlap unevenly. A language can show substantive string variation while still having a dominant form; a Wikipedia term can anchor either the dominant or an alternative cluster; a judge prompt can reduce diversity while rationales continue to express uncertainty; and one service can be a script outlier within an otherwise stable cluster.

This framework therefore separates three analytical moves:

1. **Measure degree** with exact, near-form, and substantive comparison.
2. **Describe structure** with cluster shape, service grouping, and prompt trajectory.
3. **Interpret selected cases** using external evidence, rationale claims, and language-specific research.

The first two can be automated transparently. The third remains a scholarly judgment.